In [4]:
from pathlib import Path
from datetime import datetime
from html import unescape
import re
import csv

CMS_BLOCK_PATTERN = re.compile(
    r'\[CMS ID:\s*(\d+)\](?:,\s*\[Override ID:\s*(\d+)\])?\s*:\s*',
    re.IGNORECASE
)

ROOT_PATTERNS = [
    ("agoda.reactHeader.menuViewModel", "header", "agoda.reactHeader.menuViewModel"),
    ("agoda.reactHeader.logoAndLinksMenu", "header_product_nav", "agoda.reactHeader.logoAndLinksMenu"),
    ("agoda.reactHeader.covidBanner", "covid_banner", "agoda.reactHeader.covidBanner"),
    ("window.searchBoxReact", "hero_search_box", "window.searchBoxReact"),
    ("window.flightSearchBoxReact", "flights_search_box", "window.flightSearchBoxReact"),
    ("window.carsSearchBoxReact", "cars_search_box", "window.carsSearchBoxReact"),
    ("window.homePageParams.prominentAppDownloadBanner", "app_download_banner", "window.homePageParams.prominentAppDownloadBanner"),
    ("window.homePageParams.homeComponent", "home_component", "window.homePageParams.homeComponent"),
    ("footerProps", "footer", "footerProps"),
]

SUBSECTION_RULES = [
    ("vipBarTranslations", "vip_bar", "vipBarTranslations"),
    ("loyaltyGuestBannerTranslations", "loyalty_guest_banner", "loyaltyGuestBannerTranslations"),
    ("pendingReviewsBanner", "pending_reviews_banner", "pendingReviewsBanner"),
    ("travelRestrictionBannerContainerTranslations", "travel_restriction_banner", "travelRestrictionBannerContainerTranslations"),
    ("funnelCarouselsTranslations", "funnel_carousels", "funnelCarouselsTranslations"),
    ("longStayPromotionBanner", "long_stay_banner", "longStayPromotionBanner"),
    ("featuredProperties", "featured_properties", "featuredProperties"),
    ("agodaNumbers", "agoda_numbers", "agodaNumbers"),
    ("footerSeoLinksGroups", "footer_seo_links", "footerSeoLinksGroups"),
]

def parse_filename(file_path: Path):
    name = file_path.name
    if name.endswith(".html.html"):
        name = name[:-10]
    elif name.endswith(".html"):
        name = name[:-5]

    parts = name.split("__")
    if len(parts) < 2:
        raise ValueError(f"Unexpected filename format: {file_path.name}")

    return parts[0], parts[1]

def clean_text(value: str) -> str:
    value = value.strip()
    value = value.strip('",}] ')
    value = value.replace('\\"', '"')
    value = value.replace("\\u003c", "<").replace("\\u003e", ">")
    value = value.replace("\\u0027", "'").replace("\\u0026", "&")
    value = value.replace("\\n", " ").replace("\\r", " ").replace("\\t", " ")
    value = re.sub(r"\s+", " ", value).strip()
    return value

def collect_markers(text: str):
    markers = []
    for needle, section_name, source_path in ROOT_PATTERNS:
        for m in re.finditer(re.escape(needle), text):
            markers.append((m.start(), section_name, source_path))
    for needle, section_name, source_path in SUBSECTION_RULES:
        for m in re.finditer(re.escape(needle), text):
            markers.append((m.start(), section_name, source_path))
    markers.sort(key=lambda x: x[0])
    return markers

def find_nearest_marker(markers, pos):
    best = (None, None)
    for marker_pos, section_name, source_path in markers:
        if marker_pos <= pos:
            best = (section_name, source_path)
        else:
            break
    return best

def read_value_after(text: str, start_idx: int, max_len: int = 500):
    tail = text[start_idx:start_idx + max_len]

    if not tail:
        return ""

    if tail.startswith('"'):
        escaped = False
        out = []
        for ch in tail[1:]:
            if escaped:
                out.append(ch)
                escaped = False
            elif ch == "\\":
                escaped = True
                out.append(ch)
            elif ch == '"':
                break
            else:
                out.append(ch)
        return "".join(out)

    stop_chars = [",", "}", "]", "\n", "\r", "<"]
    end = len(tail)
    for ch in stop_chars:
        idx = tail.find(ch)
        if idx != -1:
            end = min(end, idx)

    return tail[:end]

def extract_rows_from_html(file_path: Path):
    page_name, locale = parse_filename(file_path)
    text = file_path.read_text(encoding="utf-8", errors="ignore")
    text = unescape(text)

    markers = collect_markers(text)
    rows = []

    for match in CMS_BLOCK_PATTERN.finditer(text):
        cms_id = match.group(1).strip()
        override_id = match.group(2).strip() if match.group(2) else ""
        raw_text = read_value_after(text, match.end(), max_len=800)
        extracted_text = clean_text(raw_text)

        section_name, source_object_path = find_nearest_marker(markers, match.start())

        rows.append({
            "page_name": page_name,
            "locale": locale,
            "cms_id": cms_id,
            "override_id": override_id,
            "text": extracted_text,
            "section_name": section_name or "",
            "source_object_path": source_object_path or "",
        })

    return rows

def dedupe_rows(rows):
    seen = set()
    output = []

    for row in rows:
        key = (
            row["page_name"],
            row["locale"],
            row["cms_id"],
            row["override_id"],
            row["text"],
            row["section_name"],
            row["source_object_path"],
        )
        if key not in seen:
            seen.add(key)
            output.append(row)

    return output

def main():
    input_files = [
        Path("home__en-us__showcmsid.html.html"),
        Path("home__th-th__showcmsid.html.html"),
    ]

    all_rows = []
    for file_path in input_files:
        all_rows.extend(extract_rows_from_html(file_path))

    all_rows = dedupe_rows(all_rows)

    output_dir = Path(r"C:\Temp\cms_output")
    output_dir.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = output_dir / f"cms_extracted_richer_data{timestamp}.csv"

    fieldnames = [
        "page_name",
        "locale",
        "cms_id",
        "override_id",
        "text",
        "section_name",
        "source_object_path",
    ]

    with output_file.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"Saved: {output_file}")
    print(f"Total rows: {len(all_rows)}")

if __name__ == "__main__":
    main()

Saved: C:\Temp\cms_output\cms_extracted_richer_data20260610_105314.csv
Total rows: 2138
